In [2]:
# import libraries


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Load the dataset 

df = pd.read_csv('../data/AI_Workflow_Optimization_Dataset_2500_Rows_v1.csv')
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (2500, 15)


,Workflow_ID,Process_Name,Task_ID,Task_Type,Priority_Level,Department,Assigned_Employee_ID,Task_Start_Time,Task_End_Time,Estimated_Time_Minutes,Actual_Time_Minutes,Delay_Flag,Approval_Level,Employee_Workload,Cost_Per_Task
0,WF_1,Customer Complaint,TASK_1,Review,Low,Customer Service,EMP_128,2024-01-25 04:47:00,2024-01-25 08:33:00,208,226,1,Level 1,1,155.17
1,WF_2,HR Onboarding,TASK_2,Approval,Low,HR,EMP_35,2024-05-24 02:57:00,2024-05-24 06:19:00,194,202,1,Level 2,5,231.54
2,WF_3,Invoice Approval,TASK_3,Review,High,Finance,EMP_88,2024-03-22 03:34:00,2024-03-22 08:51:00,214,317,1,Level 1,8,280.95
3,WF_4,Customer Complaint,TASK_4,Review,Medium,Operations,EMP_133,2024-06-10 10:37:00,2024-06-10 13:36:00,176,179,1,Level 3,1,413.74
4,WF_5,Customer Complaint,TASK_5,Escalation,Medium,Finance,EMP_80,2024-07-02 11:38:00,2024-07-02 15:36:00,197,238,1,Level 3,3,152.30


In [3]:
# Basic info and missing values 

# Basic info and missing values
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())

# %%
# Convert date columns to datetime
date_cols = ['Task_Start_Time', 'Task_End_Time']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f"{col} range: {df[col].min()} to {df[col].max()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Workflow_ID             2500 non-null   object 
 1   Process_Name            2500 non-null   object 
 2   Task_ID                 2500 non-null   object 
 3   Task_Type               2500 non-null   object 
 4   Priority_Level          2500 non-null   object 
 5   Department              2500 non-null   object 
 6   Assigned_Employee_ID    2500 non-null   object 
 7   Task_Start_Time         2500 non-null   object 
 8   Task_End_Time           2500 non-null   object 
 9   Estimated_Time_Minutes  2500 non-null   int64  
 10  Actual_Time_Minutes     2500 non-null   int64  
 11  Delay_Flag              2500 non-null   int64  
 12  Approval_Level          2500 non-null   object 
 13  Employee_Workload       2500 non-null   int64  
 14  Cost_Per_Task           2500 non-null   

In [4]:
# Check for logical consistency: Actual_Time_Minutes should be non‑negative and comparable to Estimated_Time_Minutes
print("Any negative actual times?:", (df['Actual_Time_Minutes'] < 0).sum())
print("Any negative estimated times?:", (df['Estimated_Time_Minutes'] < 0).sum())

Any negative actual times?: 0
Any negative estimated times?: 0


In [5]:
# Delay_Flag should be 0/1 and consistent with estimated vs actual
print("Delay_Flag value counts:\n", df['Delay_Flag'].value_counts())
inconsistent_delay = ((df['Actual_Time_Minutes'] > df['Estimated_Time_Minutes']) & (df['Delay_Flag'] == 0)) | \
                     ((df['Actual_Time_Minutes'] <= df['Estimated_Time_Minutes']) & (df['Delay_Flag'] == 1))
print(f"Inconsistent delay flags: {inconsistent_delay.sum()} (will be corrected)")
# Correct: set Delay_Flag based on actual comparison
df['Delay_Flag'] = (df['Actual_Time_Minutes'] > df['Estimated_Time_Minutes']).astype(int)

Delay_Flag value counts:
 Delay_Flag
1    2373
0     127
Name: count, dtype: int64
Inconsistent delay flags: 0 (will be corrected)


In [6]:
# Check for outliers in numerical columns using IQR
num_cols = ['Estimated_Time_Minutes', 'Actual_Time_Minutes', 'Employee_Workload', 'Cost_Per_Task']
outliers_summary = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outliers_summary[col] = len(outliers)
    print(f"{col}: {len(outliers)} outliers (capped to [0, {upper:.2f}])")
    # Cap outliers (Winsorization) – set to upper/lower bounds
    df[col] = df[col].clip(lower=0, upper=upper)  # lower bound is 0 for all these


Estimated_Time_Minutes: 0 outliers (capped to [0, 352.00])
Actual_Time_Minutes: 0 outliers (capped to [0, 411.12])
Employee_Workload: 0 outliers (capped to [0, 15.50])
Cost_Per_Task: 0 outliers (capped to [0, 723.77])


In [7]:
# Data type corrections: categorical columns


cat_cols = ['Process_Name', 'Task_Type', 'Priority_Level', 'Department', 'Approval_Level']
for col in cat_cols:
    df[col] = df[col].astype('category')

# %%
# Final sanity checks
print("\nCleaned data info:")
df.info()
print("\nDescriptive statistics:")
df.describe(include='all')


Cleaned data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Workflow_ID             2500 non-null   object        
 1   Process_Name            2500 non-null   category      
 2   Task_ID                 2500 non-null   object        
 3   Task_Type               2500 non-null   category      
 4   Priority_Level          2500 non-null   category      
 5   Department              2500 non-null   category      
 6   Assigned_Employee_ID    2500 non-null   object        
 7   Task_Start_Time         2500 non-null   datetime64[ns]
 8   Task_End_Time           2500 non-null   datetime64[ns]
 9   Estimated_Time_Minutes  2500 non-null   int64         
 10  Actual_Time_Minutes     2500 non-null   int64         
 11  Delay_Flag              2500 non-null   int64         
 12  Approval_Level          2500

,Workflow_ID,Process_Name,Task_ID,Task_Type,Priority_Level,Department,Assigned_Employee_ID,Task_Start_Time,Task_End_Time,Estimated_Time_Minutes,Actual_Time_Minutes,Delay_Flag,Approval_Level,Employee_Workload,Cost_Per_Task
count,2500,2500,2500,2500,2500,2500,2500,2500,2500,2500.000000,2500.000000,2500.000000,2500,2500.000000,2500.000000
unique,2500,5,2500,5,4,5,150,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN
top,WF_2500,Purchase Order,TASK_2500,Review,Low,Operations,EMP_28,NaN,NaN,NaN,NaN,NaN,Level 3,NaN,NaN
freq,1,517,1,530,663,505,30,NaN,NaN,NaN,NaN,NaN,853,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-06-29 12:57:07.512000,2024-06-29 15:57:59.352000,123.910000,180.864000,0.949200,NaN,5.520400,277.976708
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-01-01 02:40:00,2024-01-01 05:24:00,10.000000,7.000000,0.000000,NaN,1.000000,50.320000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-27 14:20:15,2024-03-27 18:14:15,67.000000,123.000000,1.000000,NaN,3.000000,164.162500
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-06-29 23:20:30,2024-06-30 02:33:30,121.000000,180.000000,1.000000,NaN,6.000000,279.355000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-09-27 01:32:00,2024-09-27 04:50:00,181.000000,238.250000,1.000000,NaN,8.000000,388.007500
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-12-30 23:07:00,2024-12-31 02:04:00,240.000000,357.000000,1.000000,NaN,10.000000,499.940000


In [8]:
# Save cleaned dataset
df.to_csv('workflow_cleaned.csv', index=False)
print("Cleaned dataset saved as 'workflow_cleaned.csv'")

Cleaned dataset saved as 'workflow_cleaned.csv'
